## Changelog
- parent: 20260505_223808_82684e3e
- change: add ('nbhd_te', TargetEncodeColumn('Neighborhood')) step
  before AmesEncoder, on top of the GBM + MSSubClass cast pipeline
- hypothesis: Neighborhood drives a 3.3x spread in mean SalePrice
  (100k BrDale .. 335k NoRidge). One-hot makes the GBM unwind that
  signal across 24 sequential split decisions; a single numeric
  target-encoded column lets the same separation happen in one
  split. Sklearn's TargetEncoder uses internal CV during fit so
  the encoding is leakage-safe.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

    if str(comp_dir) not in sys.path:
        sys.path.insert(0, str(comp_dir))

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "eda").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

from src.eda import (
    numeric_columns,
    plot_histograms,
    plot_kde,
    plot_boxplots,
    plot_distributions,
    describe_df,
    get_missing_values,
    count_duplicates,
    plot_categorical_vs_target,
    plot_numerical_vs_target,
    target_rate_table,
    grouped_median,
)

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")


## Model

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import cross_val_score, KFold

from utils.ames_sklearn_pipeline import (
    AmesNAImputer, AmesEncoder, TargetEncodeColumn,
)
from utils.ames_feature_engineering import add_temporal_features, cast_nominal_codes

X = train_data_raw.drop(columns=["SalePrice"])
y = np.log1p(train_data_raw["SalePrice"])

# Pipeline built inline (not via build_pipeline) so the
# ('nominal', ...) and new ('nbhd_te', ...) steps are visible.
pipe = Pipeline([
    ("na",       AmesNAImputer()),
    ("fe",       FunctionTransformer(
                      add_temporal_features,
                      kw_args={"drop_originals": True},
                  )),
    ("nominal",  FunctionTransformer(cast_nominal_codes)),
    ("nbhd_te",  TargetEncodeColumn("Neighborhood")),  # NEW
    ("encoder",  AmesEncoder()),
    ("model",    GradientBoostingRegressor(
                      n_estimators=300, random_state=42,
                  )),
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipe, X, y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
rmse = -scores
print(f"CV RMSE (log-price): {rmse.mean():.4f} ± {rmse.std():.4f}")
print(f"Per-fold:            {np.round(rmse, 4).tolist()}")

pipe.fit(X, y)

In [ ]:
test_pred = np.expm1(pipe.predict(test_data_raw))

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)